# Kimi K3 extraction — train and test

This notebook annotates every dialogue in the reformatted MathDial train and test datasets using **Kimi K3**, **P11**, and maximum reasoning. Train and test run separately but use the same extraction function.

Valid cache is skipped. Missing, invalid, and unreadable cache is requested again. Results are saved as:

```text
extension/artifacts/extraction_cache/train/moonshot-direct__kimi-k3-max/P11/{dialogue_id}.json
extension/artifacts/extraction_cache/test/moonshot-direct__kimi-k3-max/P11/{dialogue_id}.json
```

## 1. Setup

Use the two `RUN_...` switches to enable or disable each split.

In [31]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

for root in [Path.cwd(), *Path.cwd().parents]:
    if (root / 'extension').is_dir() and (root / 'data').is_dir():
        os.chdir(root)
        break
else:
    raise FileNotFoundError('Run this notebook from inside the repository.')

sys.path.insert(0, str(Path.cwd()))

from extension.scripts.annotation import extraction, moonshot_kimi, prompt_loader
from extension.scripts.data_management.load_annotation_data import load_dataset

PROMPT = 'P11'
EFFORT = 'max'
MAX_WORKERS = 20
RUN_TRAIN = True
RUN_TEST = True

SLUG = moonshot_kimi.cache_slug(EFFORT)
assert PROMPT in prompt_loader.list_prompts()

try:
    moonshot_kimi._api_key()
    KEY_FOUND = True
except RuntimeError:
    KEY_FOUND = False

print(f'{SLUG} | {PROMPT} | effort={EFFORT} | workers={MAX_WORKERS}')
print('MOONSHOT_API_KEY found:', KEY_FOUND)

moonshot-direct/kimi-k3-max | P11 | effort=max | workers=20
MOONSHOT_API_KEY found: True


## 2. Load the datasets

Each API request receives one complete dialogue, including the synthetic `solution` unit and all real student turns.

In [32]:
DATA_PATHS = {
    'train': Path('data/misconception/mathdial_train.csv'),
    'test': Path('data/misconception/mathdial_test.csv'),
}

datasets = {split: load_dataset(path) for split, path in DATA_PATHS.items()}
dialogues = {
    split: extraction.dialogues_from(dataset, split=split)
    for split, dataset in datasets.items()
}

for split in DATA_PATHS:
    assert len(dialogues[split]) == datasets[split]['dialogue_id'].nunique()
    assert all(dialogue['split'] == split for dialogue in dialogues[split])

display(pd.DataFrame({
    'rows': {split: len(dataset) for split, dataset in datasets.items()},
    'dialogues': {split: len(records) for split, records in dialogues.items()},
}))

,rows,dialogues
train,15609,2253
test,3902,595


## 3. Extraction helper

The helper checks cache first and submits only dialogues that are not already valid.

In [33]:
STATUS_ORDER = ['valid', 'invalid', 'missing', 'unreadable']


def cache_audit(split):
    rows = []
    for dialogue in dialogues[split]:
        dialogue_id = dialogue['dialogue_id']
        path = extraction.cache_path(SLUG, PROMPT, dialogue_id, split)
        status, failure = 'missing', ''

        if path.is_file():
            try:
                record = json.loads(path.read_text())
                status = 'valid' if record.get('valid') else 'invalid'
                if status == 'invalid':
                    attempt = (record.get('attempts') or [{}])[-1]
                    failure = attempt.get('transport_error') or '; '.join(
                        map(str, attempt.get('errors') or [])
                    )
            except (OSError, UnicodeDecodeError, json.JSONDecodeError) as exc:
                status, failure = 'unreadable', str(exc)

        rows.append({
            'dialogue_id': dialogue_id,
            'status': status,
            'failure': failure,
        })
    return pd.DataFrame(rows)


def status_summary(audit, split):
    counts = audit['status'].value_counts().reindex(STATUS_ORDER, fill_value=0)
    result = counts.to_frame().T
    result.index = [split]
    result['total'] = len(audit)
    result['valid_rate'] = result['valid'] / result['total']
    return result[['total', *STATUS_ORDER, 'valid_rate']]


def run_extraction(split, enabled=True):
    before = cache_audit(split)
    pending_ids = set(before.loc[before['status'].ne('valid'), 'dialogue_id'])
    pending = [d for d in dialogues[split] if d['dialogue_id'] in pending_ids]

    print(f'{split}: {len(pending)} dialogue(s) require extraction')
    display(status_summary(before, split).round(3))

    if enabled and pending:
        if not KEY_FOUND:
            raise RuntimeError('Set MOONSHOT_API_KEY before extraction.')
        results = moonshot_kimi.generate_annotations(
            PROMPT,
            pending,
            reasoning_effort=EFFORT,
            max_workers=MAX_WORKERS,
        )
        display(pd.Series(results).value_counts().to_frame('dialogues'))
    elif not enabled:
        print(f'{split} extraction is disabled.')
    else:
        print(f'{split} cache is complete.')

    after = cache_audit(split)
    issues = after.loc[after['status'].ne('valid')]
    if not issues.empty:
        display(issues)
    return after

## 4. Train extraction

In [34]:
train_audit = run_extraction('train', RUN_TRAIN)

train: 0 dialogue(s) require extraction


status,total,valid,invalid,missing,unreadable,valid_rate
train,2253,2253,0,0,0,1.0


train cache is complete.


## 5. Test extraction

In [35]:
test_audit = run_extraction('test', RUN_TEST)

test: 0 dialogue(s) require extraction


status,total,valid,invalid,missing,unreadable,valid_rate
test,595,595,0,0,0,1.0


test cache is complete.


## 6. Final status

A split is complete when every dialogue is valid. Notebook 05 can then write the cached annotations into the misconception datasets.

In [36]:
final_status = pd.concat([
    status_summary(train_audit, 'train'),
    status_summary(test_audit, 'test'),
])
display(final_status.round(3))

for split, row in final_status.iterrows():
    label = 'COMPLETE' if row['valid'] == row['total'] else 'INCOMPLETE'
    print(f'{split}: {label}')

status,total,valid,invalid,missing,unreadable,valid_rate
train,2253,2253,0,0,0,1.0
test,595,595,0,0,0,1.0


train: COMPLETE
test: COMPLETE
